In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
np.set_printoptions(precision=6, suppress=True)

## Perceptron

The update is `w <- w + eta * (y - y_hat) * x`, with a step activation. AND and OR are linearly separable; XOR is not, so the single perceptron cannot converge to zero error.

In [ ]:
def perceptron_train(X, y, epochs=20, eta=0.2):
    w = np.zeros(X.shape[1], dtype=np.float64)
    b = 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            y_hat = float((xi @ w + b) >= 0.0)
            error = yi - y_hat
            w += eta * error * xi
            b += eta * error
    return w, b

X_logic = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
for name, y_logic in {"AND": np.array([0,0,0,1.]), "OR": np.array([0,1,1,1.]), "XOR": np.array([0,1,1,0.])}.items():
    w, b = perceptron_train(X_logic, y_logic)
    pred = ((X_logic @ w + b) >= 0).astype(int)
    print(name, "weights=", w, "bias=", b, "predictions=", pred, "errors=", int((pred != y_logic).sum()))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, y_logic) in zip(axes, {
    "AND": np.array([0,0,0,1.]),
    "OR": np.array([0,1,1,1.]),
    "XOR": np.array([0,1,1,0.]),
}.items()):
    w, b = perceptron_train(X_logic, y_logic)
    ax.scatter(X_logic[:, 0], X_logic[:, 1], c=y_logic, cmap="coolwarm", s=100, edgecolor="k")
    if np.linalg.norm(w) > 0:
        xs = np.linspace(-0.25, 1.25, 100)
        ys = -(w[0] * xs + b) / w[1] if abs(w[1]) > 1e-12 else np.full_like(xs, -b / w[0])
        ax.plot(xs, ys, "k--", label="decision boundary")
    ax.set_title(f"{name} perceptron")
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.25, 1.25)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.tight_layout()
plt.show()

The positive XOR points are opposite corners of the square, while the negative points occupy the other two corners. Any straight line that keeps both positive points on one side will also include at least one negative point. The perceptron can only create one linear boundary, so it needs a hidden layer or another nonlinear feature transformation to represent XOR.

## Two-layer MLP and manual backpropagation

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

def bce(y_hat, y):
    eps = 1e-12
    return -np.mean(y*np.log(y_hat+eps) + (1-y)*np.log(1-y_hat+eps))

def init_params(input_dim, hidden_dim, rng):
    return {
        "W1": rng.normal(0, 0.15, (input_dim, hidden_dim)),
        "b1": np.zeros((1, hidden_dim)),
        "W2": rng.normal(0, 0.15, (hidden_dim, 1)),
        "b2": np.zeros((1, 1)),
    }

def forward(params, X):
    z1 = X @ params["W1"] + params["b1"]
    a1 = np.maximum(z1, 0)
    z2 = a1 @ params["W2"] + params["b2"]
    return sigmoid(z2), (z1, a1)

def loss_and_grads(params, X, y):
    y_hat, (z1, a1) = forward(params, X)
    m = X.shape[0]
    dz2 = y_hat - y.reshape(-1, 1)
    grads = {
        "W2": a1.T @ dz2 / m,
        "b2": dz2.mean(axis=0, keepdims=True),
    }
    da1 = dz2 @ params["W2"].T
    dz1 = da1 * (z1 > 0)
    grads["W1"] = X.T @ dz1 / m
    grads["b1"] = dz1.mean(axis=0, keepdims=True)
    return bce(y_hat, y.reshape(-1,1)), grads

def flatten_params(params):
    return np.concatenate([params[k].ravel() for k in ("W1","b1","W2","b2")])

def numerical_gradient(params, X, y, eps=1e-5):
    numeric = {}
    for key in params:
        g = np.zeros_like(params[key])
        it = np.nditer(params[key], flags=["multi_index"], op_flags=["readwrite"])
        while not it.finished:
            idx = it.multi_index
            old = params[key][idx]
            params[key][idx] = old + eps
            plus = loss_and_grads(params, X, y)[0]
            params[key][idx] = old - eps
            minus = loss_and_grads(params, X, y)[0]
            params[key][idx] = old
            g[idx] = (plus-minus)/(2*eps)
            it.iternext()
        numeric[key] = g
    return numeric

## Gradient check

In [ ]:
rng = np.random.default_rng(SEED)
X_check = rng.normal(size=(5, 3)).astype(np.float64)
y_check = np.array([0,1,1,0,1.], dtype=np.float64)
params_check = init_params(3, 4, rng)
loss_check, analytic = loss_and_grads(params_check, X_check, y_check)
numeric = numerical_gradient(params_check, X_check, y_check)
for key in analytic:
    relative = np.linalg.norm(analytic[key]-numeric[key])/(np.linalg.norm(analytic[key])+np.linalg.norm(numeric[key])+1e-15)
    print(key, relative)
assert max(np.linalg.norm(analytic[k]-numeric[k])/(np.linalg.norm(analytic[k])+np.linalg.norm(numeric[k])+1e-15) for k in analytic) < 1e-6

## Train on make_moons

We use the same seed and full-batch vanilla gradient descent for hidden sizes 2, 8, and 32. The decision boundary illustrates why the hidden layer succeeds where the perceptron fails.

In [ ]:
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=300, noise=0.12, random_state=SEED)
X = X.astype(np.float64)

def train_numpy(X, y, hidden_dim, epochs=800, lr=0.25):
    params = init_params(X.shape[1], hidden_dim, np.random.default_rng(SEED + hidden_dim))
    losses = []
    for _ in range(epochs):
        loss, grads = loss_and_grads(params, X, y)
        losses.append(loss)
        for key in params: params[key] -= lr * grads[key]
    return params, losses

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, hidden in enumerate([2,8,32]):
    params, losses = train_numpy(X, y, hidden)
    axes[0,col].plot(losses); axes[0,col].set_title(f"hidden={hidden} loss")
    grid_x, grid_y = np.meshgrid(np.linspace(X[:,0].min()-0.5,X[:,0].max()+0.5,160), np.linspace(X[:,1].min()-0.5,X[:,1].max()+0.5,160))
    grid = np.c_[grid_x.ravel(), grid_y.ravel()]
    probs = forward(params, grid)[0].reshape(grid_x.shape)
    axes[1,col].contourf(grid_x, grid_y, probs, levels=[0,0.5,1], alpha=0.25)
    axes[1,col].scatter(X[:,0], X[:,1], c=y, cmap="coolwarm", edgecolor="k", s=12)
    axes[1,col].set_title(f"hidden={hidden} boundary")
plt.tight_layout()
plt.show()

The NumPy version makes every intermediate tensor, derivative, shape, and parameter update explicit. That is useful for understanding the computation, but it is also easy to introduce a transpose or broadcasting error. Autograd saves the repetitive derivative bookkeeping while preserving the same computational graph and gradients.